# 07 — Definitional Patterns: what do LLMs encode about "Digital Humanities"?

The earlier notebooks (02–06) examine *what the models produce* through translations, review signals, prompt comparisons, embeddings, and rationale style. This notebook turns to a different question: **what do the models think Digital Humanities *is*?** The rationale column gives us an indirect window into each model's definition of the term, and variation across models, variants, and target languages shows how unevenly they frame a concept that DH scholars themselves continue to debate.

Four motivating questions structure the analysis:

1. **Definitional registers.** What *types* of definition do LLMs reach for? Scholarly/methodological framings (interdisciplinary, computational research), compositional/literal framings (digital + humanities), human-centered framings (people, culture, experience), or generic-knowledge framings (digital + learning/knowledge)?
2. **Frontier vs. smaller-model split.** Earlier findings (xmn, Himalayan template, Llama Indus Valley template) showed that smaller models exhibit *structural* failures of multilingual representation. Do they also exhibit *conceptual* thinness — relying on the literal compositional register where frontier models reach for scholarly framings?
3. **Component asymmetry.** Which side of *Digital Humanities* is harder for the models to encode — "digital" (concrete technical concept) or "humanities" (contested scholarly field)?
4. **Definitional richness ↔ translation success.** Do rationales that reach for richer scholarly framings correlate with translations that pass the automated-review-signal suite, or is there no relationship?

**Method**: keyword-based first pass for definitional registers; component-segmented pattern matching for the digital-vs-humanities asymmetry; cross-reference with `automated_review_signals.csv` for the richness-vs-success question. The keyword approach is necessarily coarse — a future-work section flags LLM-as-judge validation on a stratified sample as the natural next step.

**Input**: `evaluation/disagreement_explorer_data.csv`, which enforces translation/rationale pairing, plus the Tier 2 decisions in `manual_exclusions.csv`.  
**Scope**: English-register variants only — `minimal`, `github_searcher`, `judge`. The `fluent_speaker` variant produces target-language rationales and is excluded from this analysis.

In [1]:
import os, sys, re
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import altair as alt

alt.data_transformers.enable('vegafusion')

sys.path.insert(0, str(Path('..').resolve()))
from scripts.utils import get_data_directory_path, get_language_family, read_csv_file, save_chart

DATA_DIR  = get_data_directory_path()
TERM      = 'Digital Humanities'
TERM_SLUG = TERM.lower().replace(' ', '_')
FIGURE_DIR = Path('..') / 'figures' / 'dhq_article'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# English-rationale variants only — fluent_speaker uses target language
ENGLISH_VARIANTS = ['minimal', 'github_searcher', 'judge']
ALL_VARIANTS     = ['minimal', 'fluent_speaker', 'github_searcher', 'judge']

LLM_SERVICES = ['claude', 'openai', 'gemini', 'deepseek', 'llama', 'gemma', 'qwen', 'mistral']
FRONTIER     = {'claude', 'openai', 'gemini', 'deepseek'}
SMALL        = {'llama', 'gemma', 'qwen', 'mistral'}

EVAL_DIR = os.path.join(DATA_DIR, 'translated_terms', TERM_SLUG, 'evaluation')
EXPLORER_CSV = os.path.join(EVAL_DIR, 'disagreement_explorer_data.csv')
EXCL_PATH = os.path.join(EVAL_DIR, 'manual_exclusions.csv')
print(f'Rationale matrix: {EXPLORER_CSV}')

Retrieving translation pipeline data directory path...

Rationale matrix: /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets/translated_terms/digital_humanities/evaluation/disagreement_explorer_data.csv


## 7.1 — Loading the rationale corpus

Build a single long-format DataFrame with one row per (service, variant, language) rationale from the paired rationale matrix used in Notebook 06. Drop empty, short, or placeholder rationales (`'No rationale provided'`, `'N/A'`, etc.), apply Tier 2 manual translation/rationale exclusions, and restrict to English-register variants. The output is the corpus that all subsequent analyses operate on.

In [2]:
PLACEHOLDER_RAT = {
    'no rationale provided', 'no rationale', 'n/a', 'none',
    'not applicable', 'no explanation provided', 'no reason provided',
    '[rationale truncated — max_tokens]',
}

def valid_rat(value):
    if not isinstance(value, str) or len(value.strip()) <= 20:
        return False
    return value.strip().rstrip('.').lower() not in PLACEHOLDER_RAT

explorer_df = read_csv_file(EXPLORER_CSV)
exclusion_lookup = {}
if os.path.exists(EXCL_PATH):
    exclusions = read_csv_file(EXCL_PATH)
    for _, row in exclusions.iterrows():
        code = str(row.get('language_code', '')).strip()
        service = str(row.get('service', '')).strip().lower()
        if not code or not service or service == 'nan':
            continue
        exclusion_lookup[(code, service)] = {
            'translation': str(row.get('exclude_translation', 'False')).strip().lower() == 'true',
            **{
                f'rationale_{variant}': str(row.get(f'exclude_rationale_{variant}', 'False')).strip().lower() == 'true'
                for variant in ALL_VARIANTS
            },
        }

rows = []
for service in LLM_SERVICES:
    for variant in ENGLISH_VARIANTS:
        rationale_col = f'{service}_rationale_{variant}'
        if rationale_col not in explorer_df.columns:
            continue
        for _, row in explorer_df[['language_code', rationale_col]].iterrows():
            code = str(row.get('language_code', '')).strip()
            rationale = row.get(rationale_col)
            decisions = exclusion_lookup.get((code, service), {})
            if (decisions.get('translation', False)
                    or decisions.get(f'rationale_{variant}', False)
                    or not valid_rat(rationale)):
                continue
            rows.append({
                'service': service,
                'variant': variant,
                'language_code': code,
                'rationale': rationale.strip(),
                'model_size': 'frontier' if service in FRONTIER else 'small',
            })
rationale_df = pd.DataFrame(rows)
print(f'Total English-register rationales: {len(rationale_df)}')
print(f'  By service:\n{rationale_df["service"].value_counts().to_string()}')

Total English-register rationales: 20661
  By service:
service
claude      2642
qwen        2635
deepseek    2626
gemma       2616
mistral     2587
gemini      2563
openai      2547
llama       2445


## 7.2 — Definitional registers (keyword pass)

Four candidate "registers" of DH definition, each operationalised as a small set of phrasal patterns. A rationale may match multiple registers (definitions overlap in practice; this isn't an exclusive classification).

- **Compositional / literal**: rationale describes DH as a combination of *digital* and *humanities* without further scholarly framing. The default fallback register.
- **Scholarly / methodological**: rationale invokes interdisciplinarity, computational methods, scholarship, academic discipline — recognising DH as a *field with internal methodological commitments*.
- **Human-centered**: rationale appeals to humans, people, culture, or human experience — emphasising the *humanities subject* of DH.
- **Generic knowledge**: rationale glosses humanities as *knowledge*, *education*, *learning* — flattening the scholarly specificity into broader cognitive terms.

The contrast that matters most for the article is whether the model treats *humanities* as **a scholarly field** (rich), **a category of human experience** (medium), or **a generic knowledge domain** (thin). The compositional register is the baseline — present in nearly every rationale — so the comparative measure is which *non-compositional* register a model reaches for.

In [3]:
REGISTER_PATTERNS = {
    'compositional_literal': [
        r'\bcombines?\b.*\bdigital\b',
        r'\bdigital\b.*\b(?:and|with|plus)\b.*\bhumanities\b',
    ],
    'scholarly_methodological': [
        r'\binterdisciplinary\b',
        r'\bcomputational\s+method',
        r'\bdigital\s+(?:methods?|tools?)\s+(?:to|in|for)\s+(?:study|stud|research|analy)',
        r'\bscholarly\b.*\bdigital\b',
        r'\bdigital\b.*\bscholarship\b',
        r'\bacademic\s+(?:field|discipline)',
    ],
    'human_centered': [
        r'\b(?:study|exploration|investigation)\s+of\s+human\s+culture',
        r'\bhuman\s+experience\b',
        r'\bpeople\b.*\bdigital\b',
        r'\bhuman\b.*\b(?:digital|computer)\b.*\b(?:cultur|stud|experience)',
    ],
    'knowledge_generic': [
        r'\bdigital\b.*\bknowledge\b',
        r'\bknowledge\b.*\bdigital\b',
        r'\bdigital\s+(?:education|learning|wisdom)',
    ],
}

def matches_register(rationale, register):
    return any(re.search(p, rationale, re.IGNORECASE) for p in REGISTER_PATTERNS[register])

# Apply all four registers
for register in REGISTER_PATTERNS:
    rationale_df[f'reg_{register}'] = rationale_df['rationale'].apply(
        lambda r: matches_register(r, register)
    )

# Service-level summary
register_summary = (
    rationale_df.groupby('service')[[f'reg_{r}' for r in REGISTER_PATTERNS]]
    .mean()
    .mul(100).round(1)
    .rename(columns={f'reg_{r}': r for r in REGISTER_PATTERNS})
)
print('Definitional register prevalence (% of rationales per service):\n')
print(register_summary.to_string())

Definitional register prevalence (% of rationales per service):

          compositional_literal  scholarly_methodological  human_centered  knowledge_generic
service                                                                                     
claude                     65.3                      34.8            15.1               34.5
deepseek                   65.6                      39.9            17.9               19.5
gemini                     69.8                      46.7            23.1               27.4
gemma                      64.4                      22.6             7.0               31.6
llama                      66.7                       7.0             8.7               10.4
mistral                    55.1                       9.4             7.0                3.9
openai                     73.0                      38.4            23.4                8.4
qwen                       56.3                      11.0             2.9                9.4


In [4]:
# Heatmap — service × register, ordered by frontier-then-small
svc_order = sorted(rationale_df['service'].unique(),
                   key=lambda s: (s not in FRONTIER, s))
reg_order = ['compositional_literal', 'scholarly_methodological', 'human_centered', 'knowledge_generic']
service_labels = {
    'claude': 'Claude', 'openai': 'OpenAI', 'gemini': 'Gemini', 'deepseek': 'DeepSeek',
    'llama': 'Llama', 'gemma': 'Gemma', 'qwen': 'Qwen', 'mistral': 'Mistral',
}
register_labels = {
    'compositional_literal': 'Compositional / literal',
    'scholarly_methodological': 'Scholarly / methodological',
    'human_centered': 'Human-centered',
    'knowledge_generic': 'Generic knowledge',
}

long = (register_summary.reset_index()
        .melt(id_vars='service', var_name='register', value_name='pct'))
long['model'] = long['service'].map(service_labels)
long['register_label'] = long['register'].map(register_labels)
svc_display_order = [service_labels[service] for service in svc_order]
reg_display_order = [register_labels[register] for register in reg_order]

heat = alt.Chart(long).mark_rect(stroke='white', strokeWidth=2).encode(
    x=alt.X('register_label:N', sort=reg_display_order, title=None,
            axis=alt.Axis(labelAngle=-30)),
    y=alt.Y('model:N', sort=svc_display_order, title=None),
    color=alt.Color('pct:Q', scale=alt.Scale(scheme='blues', domain=[0, 100]),
                    title='% of rationales'),
    tooltip=['model:N','register_label:N','pct:Q'],
).properties(width=380, height=240,
             title=alt.Title('Definitional register × service',
                             subtitle='Registers overlap; each cell is the share of a service’s rationales matching that signal.'))
txt = heat.mark_text(fontSize=10).encode(
    text=alt.Text('pct:Q', format='.0f'),
    color=alt.condition('datum.pct > 50', alt.value('white'), alt.value('black')),
)
register_chart = heat + txt
save_chart(register_chart, str(FIGURE_DIR / 'definitional_register_by_service.png'))
register_chart

alt.LayerChart(...)

## 7.3 — Component-level: digital vs humanities

The user hypothesis (and a common scholarly observation): in *Digital Humanities*, the **humanities** side is harder to translate than the **digital** side. "Digital" has settled into a stable cross-linguistic concept (computers, technology, electronic data); "humanities" is contested even within English-language academia (scholarly field? humanities-as-people? humanities-as-culture?).

I operationalize this by asking whether a rationale contains an explicit **gloss signal** for either component. A digital signal uses vocabulary such as *technology*, *computational*, *electronic*, or *computer-based*. A humanities signal uses vocabulary such as *scholarship*, *humanistic*, *liberal arts*, *interdisciplinary*, *cultural studies*, or more generic descriptions of humanities fields, knowledge, culture, and experience. I then compare the share of each service's rationales matching the two signal sets.

**How the code determines a gloss:**

- Matching is case-insensitive and limited to the English-register rationales. The regex word boundaries reduce substring matches but do not perform syntactic or semantic parsing.
- `DIGITAL_GLOSS` is intentionally broad. A rationale matches if it invokes computers, computation, electronics, information technology, or technological tools/methods. Because several patterns can match anywhere in the rationale, this means the model uses a technical frame; it does not prove that the relevant phrase directly defines the word *digital*.
- Humanities evidence is split into two layers. `HUMANITIES_GLOSS_SCHOLARLY` looks for field-specific language such as *scholarship*, *liberal arts*, *cultural studies*, or *academic discipline*. `HUMANITIES_GLOSS_GENERIC` captures broader language about humanities fields, knowledge, culture, society, history, or human experience.
- `has_humanities_gloss_any` is the logical OR of those scholarly and generic layers. I use this combined measure in the digital-versus-humanities chart so that humanities is not penalized simply because its explanations are conceptually more varied. The component table retains both layers separately.
- The digital and humanities indicators are independent. One rationale can match both, either, or neither. Percentages therefore describe prevalence among all retained rationales and should not be added together.
- The asymmetry statistic is `digital percentage - humanities-any percentage`. A positive value means the service invokes digital vocabulary more often; a negative value means it invokes humanities vocabulary more often. It measures differential explicitness under these pattern lists, not which component is objectively harder to translate.
- Absence of a match is not evidence that the model lacks the concept. Paraphrases outside the closed pattern lists become false negatives, while incidental mentions, negation, or unrelated uses can become false positives. These are theory-laden screening rules for aggregate comparison, not validated semantic annotations.

In [5]:
DIGITAL_GLOSS = [
    r'\bdigital\s+(?:technology|technologies|tools?|methods?|techniques?)\b',
    r'\bcomput(?:er|ational|ing)\b',
    r'\belectronic\b',
    r'\binformation\s+technology\b',
    r'\btechnolog(?:y|ies|ical)\b',
]

HUMANITIES_GLOSS_SCHOLARLY = [
    r'\b(?:liberal\s+arts|scholarly|scholarship|humanistic|humanist)\b',
    r'\b(?:cultural\s+studies|literary\s+studies|historical\s+studies)\b',
    r'\b(?:academic\s+field|academic\s+discipline)\b',
    r'\binterdisciplinary\s+(?:field|study|research|approach)\b',
]

HUMANITIES_GLOSS_GENERIC = [
    r'\bhumanities\s+(?:fields?|subjects?|disciplines?|areas?)\b',
    r'\bhuman(?:istic|ities)\s+(?:knowledge|topics|matters)\b',
    r'\bhuman\s+(?:culture|experience|society|history)\b',
]

def _any(rat, pats):
    return any(re.search(p, rat, re.IGNORECASE) for p in pats)

rationale_df['has_digital_gloss']            = rationale_df['rationale'].apply(lambda r: _any(r, DIGITAL_GLOSS))
rationale_df['has_humanities_gloss_scholar'] = rationale_df['rationale'].apply(lambda r: _any(r, HUMANITIES_GLOSS_SCHOLARLY))
rationale_df['has_humanities_gloss_generic'] = rationale_df['rationale'].apply(lambda r: _any(r, HUMANITIES_GLOSS_GENERIC))
rationale_df['has_humanities_gloss_any']     = rationale_df['has_humanities_gloss_scholar'] | rationale_df['has_humanities_gloss_generic']

component_summary = (
    rationale_df.groupby('service')[['has_digital_gloss',
                                      'has_humanities_gloss_scholar',
                                      'has_humanities_gloss_generic',
                                      'has_humanities_gloss_any']]
    .mean().mul(100).round(1)
)
print('Component-gloss signal prevalence (% of rationales):\n')
print(component_summary.to_string())
print('\nDigital-vs-humanities-any asymmetry (digital minus humanities, percentage points):')
print((component_summary['has_digital_gloss'] - component_summary['has_humanities_gloss_any']).round(1).to_string())

Component-gloss signal prevalence (% of rationales):

          has_digital_gloss  has_humanities_gloss_scholar  has_humanities_gloss_generic  has_humanities_gloss_any
service                                                                                                          
claude                 37.5                          40.3                          12.7                      46.5
deepseek               39.5                          38.1                          14.6                      45.5
gemini                 42.8                          44.1                          27.2                      57.1
gemma                  48.8                          16.2                          18.3                      31.2
llama                  34.7                          11.0                          14.6                      24.3
mistral                19.6                           7.1                           6.1                      12.5
openai                 56.3       

In [6]:
# Side-by-side bars: digital vs humanities gloss per service
comp_long = (component_summary[['has_digital_gloss', 'has_humanities_gloss_any']]
             .reset_index()
             .melt(id_vars='service', var_name='component', value_name='pct'))
comp_long['component'] = comp_long['component'].map({
    'has_digital_gloss': 'digital',
    'has_humanities_gloss_any': 'humanities (any gloss)',
})
comp_long['model'] = comp_long['service'].map(service_labels)

component_chart = alt.Chart(comp_long).mark_bar().encode(
    x=alt.X('pct:Q', title='% of rationales matching gloss signal'),
    y=alt.Y('model:N', sort=svc_display_order, title=None),
    color=alt.Color('component:N', title='Component',
                    scale=alt.Scale(domain=['digital', 'humanities (any gloss)'],
                                    range=['#1976d2', '#c0680c'])),
    yOffset='component:N',
    tooltip=['model:N','component:N','pct:Q'],
).properties(width=420, height=300,
             title=alt.Title('Component-gloss signals across services',
                             subtitle='Regex prevalence; signals indicate explicit vocabulary, not validated definitions.'))
save_chart(component_chart, str(FIGURE_DIR / 'digital_humanities_component_gloss_by_service.png'))
component_chart

alt.Chart(...)

## 7.4 — Variant effects on the definitional register

Does the `github_searcher` framing ("you are building a multilingual search corpus") shift the model's definitional register away from scholarly framing and toward something more pragmatic — corpus-building, search queries, topic tags? Does the `judge` variant aggregate scholarly framing from prior variants? This section cross-tabs register × variant within each service.

In [7]:
# Within-service variant effect on scholarly register
variant_register = (
    rationale_df.groupby(['service', 'variant'])[[f'reg_{r}' for r in REGISTER_PATTERNS]]
    .mean().mul(100).round(1)
    .rename(columns={f'reg_{r}': r for r in REGISTER_PATTERNS})
    .reset_index()
)
print('Definitional register prevalence by service and prompt (%):')
print(variant_register.to_string(index=False))

# Long format for plot
var_long = variant_register.melt(
    id_vars=['service','variant'],
    value_vars=['scholarly_methodological','human_centered','knowledge_generic'],
    var_name='register', value_name='pct',
)
variant_labels = {'minimal': 'Minimal', 'github_searcher': 'GitHub searcher', 'judge': 'Judge'}
variant_display_order = [variant_labels[variant] for variant in ENGLISH_VARIANTS]
var_long['prompt'] = var_long['variant'].map(variant_labels)
var_long['model'] = var_long['service'].map(service_labels)
var_long['register_label'] = var_long['register'].map(register_labels)

variant_chart = alt.Chart(var_long).mark_bar().encode(
    x=alt.X('prompt:N', sort=variant_display_order, title=None),
    y=alt.Y('pct:Q', title='% of rationales'),
    color=alt.Color('register_label:N', title='Definitional register',
                    scale=alt.Scale(domain=[register_labels['scholarly_methodological'], register_labels['human_centered'], register_labels['knowledge_generic']],
                                    range=['#1a6b30','#c0680c','#5a4a8c'])),
    xOffset=alt.XOffset('register_label:N'),
    column=alt.Column('model:N', sort=svc_display_order, title=None,
                       header=alt.Header(labelOrient='top', titleOrient='top')),
).properties(width=85, height=180,
             title=alt.Title('Variant effect on definitional register',
                             subtitle='Grouped bars because the three keyword registers can overlap within one rationale.'))
save_chart(variant_chart, str(FIGURE_DIR / 'definitional_register_by_service_prompt.png'))
variant_chart

Definitional register prevalence by service and prompt (%):
 service         variant  compositional_literal  scholarly_methodological  human_centered  knowledge_generic
  claude github_searcher                   77.3                      47.4            13.9               29.8
  claude           judge                   63.0                      37.1            18.3               33.9
  claude         minimal                   55.6                      19.9            13.1               39.8
deepseek github_searcher                   58.5                      34.9             6.5                9.1
deepseek           judge                   72.3                      33.7            19.5               25.7
deepseek         minimal                   66.0                      51.1            27.7               23.7
  gemini github_searcher                   61.4                      28.9            11.9               20.4
  gemini           judge                   91.8                     

alt.Chart(...)

## 7.5 — Cross-reference with language-level review burden

This exploratory cross-reference asks whether scholarly framing appears more often on languages with fewer automated review signals. Each rationale inherits its language row's aggregate `flag_count` from `automated_review_signals.csv`; the count summarizes signals across services and is not a quality score for the individual service translation attached to that rationale. I compute mean language-level flag count by register as an indicator of review burden, not translation success.

**Interpretive limit**: this is associational and ecologically aggregated. A scholarly rationale and a lower language-level flag count may both reflect stronger digital resources for that language, while service-level failures elsewhere still contribute to the same flag count.

In [8]:
qf_path = os.path.join(DATA_DIR, 'translated_terms', TERM_SLUG, 'evaluation', 'automated_review_signals.csv')
qf = pd.read_csv(qf_path, dtype=str)
qf['flag_count'] = qf['flag_count'].astype(int)

rationale_q = rationale_df.merge(
    qf[['language_code', 'flag_count', 'language_family']],
    on='language_code', how='left',
)
rationale_q['flag_count'] = rationale_q['flag_count'].fillna(0).astype(int)

# Mean flag count by register
for register in REGISTER_PATTERNS:
    col = f'reg_{register}'
    a = rationale_q.loc[ rationale_q[col], 'flag_count'].mean()
    b = rationale_q.loc[~rationale_q[col], 'flag_count'].mean()
    print(f'  {register:30s}: mean flags  with={a:.2f}  without={b:.2f}  diff={a-b:+.2f}')

  compositional_literal         : mean flags  with=3.95  without=3.96  diff=-0.02
  scholarly_methodological      : mean flags  with=3.75  without=4.02  diff=-0.27
  human_centered                : mean flags  with=4.01  without=3.94  diff=+0.07
  knowledge_generic             : mean flags  with=4.16  without=3.91  diff=+0.26


In [9]:
# Per-service: mean flag count by scholarly-register status
svc_register_flags = (
    rationale_q.groupby(['service', 'reg_scholarly_methodological'])['flag_count']
    .mean().reset_index()
    .rename(columns={'reg_scholarly_methodological': 'has_scholarly'})
)
svc_register_flags['has_scholarly'] = svc_register_flags['has_scholarly'].map({True: 'scholarly', False: 'not scholarly'})
svc_register_flags['model'] = svc_register_flags['service'].map(service_labels)
print('Mean automated flag count by service and scholarly-register status:')
print(svc_register_flags.to_string(index=False))

quality_chart = alt.Chart(svc_register_flags).mark_bar().encode(
    x=alt.X('flag_count:Q', title='Mean language-level flag count'),
    y=alt.Y('model:N', sort=svc_display_order, title=None),
    color=alt.Color('has_scholarly:N', title='Rationale register',
                    scale=alt.Scale(domain=['scholarly', 'not scholarly'],
                                    range=['#1a6b30', '#aaaaaa'])),
    yOffset='has_scholarly:N',
    tooltip=['model:N','has_scholarly:N','flag_count:Q'],
).properties(width=420, height=300,
             title=alt.Title('Language-level review burden by rationale register',
                             subtitle='Flag counts aggregate signals across services; they do not score the individual rationale or translation.'))
save_chart(quality_chart, str(FIGURE_DIR / 'scholarly_rationale_register_and_review_flags.png'))
quality_chart

Mean automated flag count by service and scholarly-register status:
 service has_scholarly  flag_count    model
  claude not scholarly    4.024376   Claude
  claude     scholarly    3.895539   Claude
deepseek not scholarly    4.153262 DeepSeek
deepseek     scholarly    3.695320 DeepSeek
  gemini not scholarly    3.994139   Gemini
  gemini     scholarly    3.913189   Gemini
   gemma not scholarly    3.973827    Gemma
   gemma     scholarly    3.972927    Gemma
   llama not scholarly    3.873407    Llama
   llama     scholarly    3.911765    Llama
 mistral not scholarly    3.956077  Mistral
 mistral     scholarly    3.913223  Mistral
  openai not scholarly    4.233758   OpenAI
  openai     scholarly    3.428864   OpenAI
    qwen not scholarly    4.069906     Qwen
    qwen     scholarly    3.242215     Qwen


alt.Chart(...)

## 7.6 — Limitations and future LLM-as-judge validation

The keyword approach is *coarse*: it relies on closed lists of phrasal patterns that are inherently incomplete and biased toward English-language scholarly conventions of how DH is described. Several rationales that reach a scholarly register through different phrasing (e.g., *"intersection of the humanities and computational methods"*) may slip past the regex. Conversely, a rationale that mentions *"interdisciplinary"* about something unrelated to the field's identity would be miscounted as scholarly.

A natural validation step is **LLM-as-judge classification** on a stratified sample: take ~200 rationales sampled across services × variants × registers, present each to a separate LLM (Claude or GPT-4o) with a definitional-register classification schema, and compare the LLM's judgement to the keyword classifier's output. A high inter-rater agreement validates the keyword approach for paper-quality results; a low agreement flags specific register categories that need refinement.

**Other natural extensions:**

- **Family-level analysis**: does the model's definitional richness vary by *target* language family? Does Claude reach for scholarly framing more often when translating into Indo-European languages than into Niger-Congo languages?
- **Within-rationale shifts**: does the model start with a scholarly definition and then drift to a literal compositional one as it discusses target-language morphology? Sentence-level tagging would surface this.
- **Component asymmetry within a single sentence**: do the rationales that gloss *digital* also gloss *humanities*, or is the asymmetry happening *within* rationales (not across)?
- **Cross-reference with the success cases from §1.5**: in the languages where multiple models converge on the same translation (Māori `matihiko`, Corsican `umanità digitale`), do the rationales also converge on the same definitional register? If yes, this would strengthen the claim that scholarly definitions are a precondition for translation convergence.